# 01 — Exercises: regression

The housing examples are synthetic. Attempt the written prompts before opening `solutions.ipynb`. The starter code runs from top to bottom; complete the marked modeling steps before interpreting the starter baseline plots and scores.

## Setup

The test rows are held aside before model fitting. The preprocessing transformer is defined once and will be fit only through a pipeline.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def find_data_file(filename):
    for directory in (Path.cwd(), *Path.cwd().parents):
        data_path = directory / "data" / filename
        if data_path.is_file():
            return data_path
    raise FileNotFoundError(f"Could not find data/{filename}")


housing_data = pd.read_csv(find_data_file("housing.csv"))
feature_data = housing_data.drop(columns="price")
price_response = housing_data["price"]
training_features, test_features, training_response, test_response = train_test_split(
    feature_data, price_response, test_size=0.25, random_state=27
)
numeric_columns = ["area_m2", "bedrooms", "age_years", "distance_km"]
categorical_columns = ["neighborhood"]
preprocessing = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            numeric_columns,
        ),
        (
            "category",
            Pipeline(
                steps=[
                    ("impute", SimpleImputer(strategy="most_frequent")),
                    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_columns,
        ),
    ]
)

## Exercise 1 — Fit a linear model

Fit `LinearRegression` in the pipeline. Report held-out MAE, RMSE, and $R^2$, then compare MAE with the training-median baseline. Does the model improve on this split?

In [ ]:
# Complete the marked steps to fit the pipeline and evaluate its predictions.
regression_model = Pipeline(
    steps=[("preprocessing", preprocessing), ("model", LinearRegression())]
)

median_baseline = DummyRegressor(strategy="median")
baseline_training_input = np.zeros((len(training_response), 1))
baseline_test_input = np.zeros((len(test_response), 1))
median_baseline.fit(baseline_training_input, training_response)
baseline_predictions = median_baseline.predict(baseline_test_input)

# TODO: fit regression_model on training_features and training_response.
# Then replace the starter with: regression_model.predict(test_features).
test_predictions = baseline_predictions

print("Starter MAE:", mean_absolute_error(test_response, test_predictions))
print("Starter RMSE:", mean_squared_error(test_response, test_predictions) ** 0.5)
print("Starter R²:", r2_score(test_response, test_predictions))
print("Compare your fitted model against the median baseline.")

## Exercise 2 — Diagnose errors

After fitting the model, plot residuals against predictions. Is there visible curvature or changing spread? Give one reason not to overinterpret a small test sample.

In [ ]:
# This diagnostic uses `test_predictions`. Fit the model in the previous cell
# before interpreting this plot; the initial plot shows the baseline.
test_residuals = test_response.to_numpy() - test_predictions
plt.scatter(test_predictions, test_residuals, alpha=0.75)
plt.axhline(0, color="black", linestyle="--")
plt.xlabel("Predicted price")
plt.ylabel("Observed minus predicted")
plt.title("Held-out residuals")

## Exercise 3 — Explain the metric and model limits

1. One prediction misses by 100,000 and nine miss by 5,000. Why does RMSE react more strongly to the large miss than MAE?
2. If training areas range from 40 to 180 m², what assumption does a 400 m² prediction require?
3. Why does a neighborhood coefficient not identify the effect of moving the same home to that neighborhood?

## Exercise 4 — Select Ridge regularization

Compare $\alpha\in\{0.1,1,10,100\}$ by five-fold CV RMSE on training rows. Put preprocessing inside `GridSearchCV`. Select without consulting the final test score. Explain why repeated test tuning is problematic.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold

# TODO: put Ridge in a pipeline with `preprocessing`; search alpha values
# using training data and neg_root_mean_squared_error.
print("Candidate alpha values:", [0.1, 1.0, 10.0, 100.0])
print("Training rows available for cross-validation:", len(training_features))

## Exercise 5 — Short report

Include test sample size, MAE and RMSE, baseline comparison, one residual observation, and one limitation. Explicitly describe the data as synthetic. Do not claim causation or accuracy on real housing data.